# Module 14 Assignment
## NOTE: All of the code that you write must be in PySpark for this assignment. https://spark.apache.org/docs/latest/api/python/reference/pyspark.pandas/index.html

## Install Spark

Install Dependencies:

1.   Java 8
2.   Apache Spark with hadoop and
3.   Findspark (used to locate the spark in the system)

In [1]:
!apt-get install openjdk-8-jdk-headless -qq > /dev/null
!wget -q http://archive.apache.org/dist/spark/spark-3.1.1/spark-3.1.1-bin-hadoop3.2.tgz
!tar xf spark-3.1.1-bin-hadoop3.2.tgz
!pip install -q findspark

Set Enviorment Variables:

In [2]:
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
os.environ["SPARK_HOME"] = "/content/spark-3.1.1-bin-hadoop3.2"

In [3]:
!ls

sample_data  spark-3.1.1-bin-hadoop3.2	spark-3.1.1-bin-hadoop3.2.tgz


In [4]:
import findspark
findspark.init()
from pyspark.sql.functions import *
from pyspark.sql import SparkSession

spark = SparkSession.builder.master("local[*]").appName("Mod14").config('spark.ui.port', '4050').getOrCreate()
spark.conf.set("spark.sql.repl.eagerEval.enabled", True) # Property used to format output tables better
spark

## Load the Dataset

Documentation:

- https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page
- https://www.nyc.gov/assets/tlc/downloads/pdf/data_dictionary_trip_records_hvfhs.pdf

In [5]:
!wget -P ./taxi-files/ https://d37ci6vzurychx.cloudfront.net/trip-data/fhvhv_tripdata_2020-03.parquet

--2025-05-03 19:40:49--  https://d37ci6vzurychx.cloudfront.net/trip-data/fhvhv_tripdata_2020-03.parquet
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 108.138.245.225, 108.138.245.96, 108.138.245.16, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|108.138.245.225|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 346478514 (330M) [application/x-www-form-urlencoded]
Saving to: ‘./taxi-files/fhvhv_tripdata_2020-03.parquet’

fhvhv_tripdata_2020 100%[===================>] 330.43M  28.9MB/s    in 12s     

2025-05-03 19:41:02 (26.6 MB/s) - ‘./taxi-files/fhvhv_tripdata_2020-03.parquet’ saved [346478514/346478514]



## (10 points) Check to make sure necessary files were downloaded
The command below should return '1' - if it does not, try 'Disconnect and delete runtime' and re-run all code above

Note: We are only focusing on the 'FHVHV' subset of the NYC Taxi data - high-volume for-hire vehicle bases for companies dispatching 10,000+ trip per day, meaning Uber, Lyft, Via, and Juno

In [6]:
!find ./taxi-files/ -maxdepth 1 -name "*fhvhv_tripdata*" -printf '.' | wc -m

1


## Load the Parquet Files with Spark

In [7]:
fhvhv_data = spark.read.format("parquet").load("./taxi-files/*").limit(100000)

In [8]:
fhvhv_data.show()

+-----------------+--------------------+--------------------+-------------------+-------------------+-------------------+-------------------+------------+------------+----------+---------+-------------------+-----+----+---------+--------------------+-----------+----+----------+-------------------+-----------------+------------------+----------------+--------------+
|hvfhs_license_num|dispatching_base_num|originating_base_num|   request_datetime|  on_scene_datetime|    pickup_datetime|   dropoff_datetime|PULocationID|DOLocationID|trip_miles|trip_time|base_passenger_fare|tolls| bcf|sales_tax|congestion_surcharge|airport_fee|tips|driver_pay|shared_request_flag|shared_match_flag|access_a_ride_flag|wav_request_flag|wav_match_flag|
+-----------------+--------------------+--------------------+-------------------+-------------------+-------------------+-------------------+------------+------------+----------+---------+-------------------+-----+----+---------+--------------------+-----------+--

## Data Cleaning


### Q1 (10 points)
Remove any duplicate rows.

Output the remaining number of rows

In [9]:

fhvhv_data_cleaned = fhvhv_data.dropDuplicates()
remaining_rows = fhvhv_data_cleaned.count()
print(f"Remaining number of rows: {remaining_rows}")

Remaining number of rows: 100000


### Q2 (10 points)
Remove any trips with a duration of less than 1 minute or more than 24 hours

Output the remaining number of rows

In [10]:
fhvhv_data_filtered = fhvhv_data_cleaned.filter((col('trip_time') >= 60) & (col('trip_time') <= 86400))
remaining_rows_after_filter = fhvhv_data_filtered.count()
print(f"Remaining number of rows after filtering: {remaining_rows_after_filter}")

Remaining number of rows after filtering: 99963


### Q3 (10 points)
Remove any trips with base_passenger_fare less than \\$2.50  or greater than  \$1,000

Output the remaining number of rows

In [11]:
fhvhv_data_filtered_fare = fhvhv_data_filtered.filter((col('base_passenger_fare') >= 2.50) & (col('base_passenger_fare') <= 1000))
remaining_rows_after_fare_filter = fhvhv_data_filtered_fare.count()
print(f"Remaining number of rows after filtering base_passenger_fare: {remaining_rows_after_fare_filter}")

Remaining number of rows after filtering base_passenger_fare: 99169


## Data Transformation

### Q4 (20 points)
Create three new columns:
 - A column that indicates whether the trip was taken during the morning rush hour (6:00-9:59 AM), the afternoon rush hour (3:00-6:59 PM), or outside of rush hour.
 - A column that calculates the total amount of the trip (fare amount + tip amount)
 - A column that calculates the speed of the trip (distance / duration).


 Show the first 5 results

In [12]:
from pyspark.sql.functions import hour, when, col

# Create the "rush_hour" column based on pickup_datetime
fhvhv_data_with_columns = fhvhv_data_filtered_fare.withColumn(
    "rush_hour",
    when((hour(col('pickup_datetime')) >= 6) & (hour(col('pickup_datetime')) < 10), "Morning Rush Hour")
    .when((hour(col('pickup_datetime')) >= 15) & (hour(col('pickup_datetime')) < 19), "Afternoon Rush Hour")
    .otherwise("Outside Rush Hour")
)

# Create the "total_amount" column (fare + tip)
fhvhv_data_with_columns = fhvhv_data_with_columns.withColumn(
    "total_amount",
    col("base_passenger_fare") + col("tips")
)

# Create the "speed" column (distance / duration, assuming trip_time is in seconds)
fhvhv_data_with_columns = fhvhv_data_with_columns.withColumn(
    "speed",
    col("trip_miles") / (col("trip_time") / 3600)
)

fhvhv_data_with_columns.select("pickup_datetime", "rush_hour", "total_amount", "speed").show(5)

+-------------------+-----------------+------------+------------------+
|    pickup_datetime|        rush_hour|total_amount|             speed|
+-------------------+-----------------+------------+------------------+
|2020-03-01 00:03:40|Outside Rush Hour|       24.45| 25.98665554628857|
|2020-03-01 00:28:05|Outside Rush Hour|       11.88|19.452147239263805|
|2020-03-01 00:03:07|Outside Rush Hour|       14.57|20.435146443514647|
|2020-03-01 00:18:42|Outside Rush Hour|       13.89|14.190000000000001|
|2020-03-01 00:44:24|Outside Rush Hour|        20.2|16.869767441860464|
+-------------------+-----------------+------------+------------------+
only showing top 5 rows



### Q5 (10 points)
The 'hvfhs_license_num' column corresponds to the different HVFHS businesses:
- HV0002: Juno
- HV0003: Uber
- HV0004: Via
- HV0005: Lyft

Create a new column that contains the business name based on the license number.  
Output a count of how many rides each business has in the data.

In [13]:
# Create the "business_name" column based on "hvfhs_license_num"
fhvhv_data_with_business = fhvhv_data_with_columns.withColumn(
    "business_name",
    when(col("hvfhs_license_num") == "HV0002", "Juno")
    .when(col("hvfhs_license_num") == "HV0003", "Uber")
    .when(col("hvfhs_license_num") == "HV0004", "Via")
    .when(col("hvfhs_license_num") == "HV0005", "Lyft")
    .otherwise("Unknown")
)

# Group by "business_name" and count the number of rides for each business
ride_counts = fhvhv_data_with_business.groupBy("business_name").count()

ride_counts.show()

+-------------+-----+
|business_name|count|
+-------------+-----+
|         Lyft|24546|
|         Uber|73222|
|          Via| 1401|
+-------------+-----+



### Q6 (20 points)
We want to know the average driver pay and rider cost per mile for each business.

Calculate the average driver_pay including tips by trip_miles and total cost (base_passenger_fare + tolls + bcf + sales_tax + congestion_surcharge + airport_fee) by trip_miles, grouped by business

Show the result

In [14]:
# Calculate total driver pay (including tips) per trip
fhvhv_data_with_business = fhvhv_data_with_business.withColumn(
    "total_driver_pay",
    col("driver_pay") + col("tips")
)

# Calculate total cost per trip (sum of relevant cost columns)
fhvhv_data_with_business = fhvhv_data_with_business.withColumn(
    "total_cost",
    col("base_passenger_fare") + col("tolls") + col("bcf") + col("sales_tax") + col("congestion_surcharge") + col("airport_fee")
)

# Calculate average driver pay per mile and average total cost per mile, grouped by business
result = fhvhv_data_with_business.groupBy("business_name").agg(
    (sum("total_driver_pay") / sum("trip_miles")).alias("avg_driver_pay_per_mile"),
    (sum("total_cost") / sum("trip_miles")).alias("avg_total_cost_per_mile")
)

result.show()

+-------------+-----------------------+-----------------------+
|business_name|avg_driver_pay_per_mile|avg_total_cost_per_mile|
+-------------+-----------------------+-----------------------+
|         Lyft|     2.5951632275743792|                   null|
|         Uber|     3.4108818787296555|   1.745166321049562E-4|
|          Via|    0.34788077579200993|                   null|
+-------------+-----------------------+-----------------------+



## Saving Data

### Q7 (10 points)
Save your results from the previous question as a SINGLE csv file

In [15]:
result.write \
    .option("header", "true").csv("/content/MHito-module14.csv")

## What to turn in
- Download the .ipynb (File > Download) and upload to Canvas as \<student-first-initial\>\<student-lastname\>-module14.ipynb
- Upload a HTML version of the notebook with all results visible as \<student-first-initial\>\<student-lastname\>-module14.html
- Upload the single csv file from the last question as \<student-first-initial\>\<student-lastname\>-module14.csv

Do not zip the files